In [ ]:
%%capture
import os
from pathlib import Path

import pandas as pd
from dj_notebook import activate

env_file = os.environ["META_ENV"]
reports_folder = Path(os.environ["META_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["META_ANALYSIS_FOLDER"])
pharmacy_folder = Path(os.environ["META_PHARMACY_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option("future.no_silent_downcasting", True)

In [ ]:
from meta_ae.models import AeInitial, AeTmg, AeFinalClassification
from edc_model_to_dataframe.read_frame_edc import read_frame_edc


In [ ]:
import re

from clinicedc_constants import NOT_APPLICABLE, OTHER

AE_CLASSIFICATION = "edc_adverse_event.aeclassification"


def normalized(value: str | None) -> str:
    """Casefold, collapse whitespace, treat `_` as a space."""
    return " ".join((value or "").replace("_", " ").split()).casefold()


def get_terms(
    list_data: dict,
    list_model: str = AE_CLASSIFICATION,
    aliases: dict[str, str] | None = None,
) -> list[tuple[str, str]]:
    """Every term worth looking for, longest first, as (term, name).

    Both the name and the display_name are terms, so `lactic_acidosis`
    and "Lactic acidosis" each match. Longest first so a term that
    contains another wins: "hepatomegaly with steatosis" is matched
    before "hepatomegaly" can claim it.
    """
    names = [name for name, _ in list_data[list_model]]
    terms: set[tuple[str, str]] = set()
    for name, display_name in list_data[list_model]:
        if name in [OTHER, NOT_APPLICABLE]:
            continue
        terms |= {(normalized(name), name), (normalized(display_name), name)}
    for term, name in (aliases or {}).items():
        if name not in names:
            raise ValueError(f"Alias points at a name not on the list. Got {name}.")
        terms.add((normalized(term), name))
    return sorted((t for t in terms if t[0]), key=lambda t: len(t[0]), reverse=True)


def get_classification_name(
    list_data: dict,
    value: str | None,
    list_model: str = AE_CLASSIFICATION,
    aliases: dict[str, str] | None = None,
) -> str | None:
    """Return the list `name` named in `value`, or None.

    "Anaemia grade 3" -> "anaemia"

    Matches whole words only, so "anaemia" is not found inside
    "hypoanaemial". Misspellings are not the list's to know about:
    pass them in `aliases` as {"Anemia": "anaemia"}.
    """
    text = normalized(value)
    if not text:
        return None
    for term, name in get_terms(list_data, list_model, aliases):
        if re.search(rf"\b{re.escape(term)}\b", text):
            return name
    return None

In [ ]:
df = read_frame_edc("meta_ae.AeFinalClassification", drop_sys_columns=True)

In [ ]:
df.ae_classification_other.value_counts().to_frame().reset_index().ae_classification_other.to_list()

In [ ]:
from meta_ae.list_data import list_data

aliases = {
    "dyslipidemia": "dyslipidaemia",
    "dsylipidemia": "dyslipidaemia",
    "anemia": "anaemia",
    "renal insufficiency": "renal_insufficiency",
    "renal insufficency": "renal_insufficiency",
    "renal insuficiency": "renal_insufficiency",
    "renal insuffeciency": "renal_insufficiency",
    "death": "death",
    "hyperlipidemia": "hyperlipidaemia",
    "hyperamylasemia": "hyperamylasaemia",
    "hyperuricemia": "hyperuricaemia",
    "hypoalbuminemia": "hypoalbuminaemia",
    "liver insufficiency": "liver_insufficiency",
    "liver insufficency": "liver_insufficiency",
    "leukopenia": "leukopenia",
    "pancytopenia": "pancytopenia",
    "thrombocytopenia": "thrombocytopenia",
}


for other_value in df.ae_classification_other.value_counts().to_frame().reset_index().ae_classification_other.to_list():
    try:
        get_classification_name(list_data, other_value, aliases=aliases)
    except ValueError as e:
        print(e)
